# Rendille Translation Pipeline — Google Colab
**Speech-to-speech translation: Rendille → English**

This notebook trains the full pipeline on Google Colab's free GPU.
**Estimated time:** 2–4 hours (including model training)

---

**What this does:**
1. Clones the GitHub repo
2. Installs Python dependencies
3. Downloads Rendille–English Bible parallel corpus
4. Fine-tunes NLLB-200 translation model (LoRA)
5. Launches interactive Gradio demo

**Note:** ASR (speech recognition) requires Rendille audio Bible data.
Once you have the audio from MegaVoice, run cells in `asr/` section.

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install Python dependencies FIRST (before any other imports)
!pip install torch transformers datasets sacrebleu peft accelerate sentencepiece gradio -q

# Verify installation
import torch
from transformers import AutoModelForSeq2SeqLM
print(f'✓ PyTorch: {torch.__version__}')
print(f'✓ CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print('✓ All packages installed')


In [ ]:
# Clone repository
!git clone https://github.com/KulmichaBullo/rendille-translator.git
%cd rendille-translator
!ls -la


In [ ]:
# Quick environment verification
import sys
sys.path.append('.')
from scripts.setup_checklist import check_env
check_env()


In [ ]:
# Download Rendille–English Bible corpus
# Uses multiple fallback strategies (CDN mirrors + HF)
!python scripts/download_bible.py


In [ ]:
# MANUAL UPLOAD FALLBACK (if automatic download fails)\n#\n# If the cell above failed, manually upload the 4 Bible files:\n# 1. Download from https://ebible.org/ (search "Rendille")\n# 2. Also download English (World English Bible = 'eng') from same site\n# 3. Upload to Colab:\n#    - Click the 📁 folder icon on left sidebar\n#    - Click "Upload" and select the 4 files\n#    - Files must be named exactly:\n#      • rel_vref.txt  (Rendille verse references)\n#      • rel_extract.txt  (Rendille verse text)\n#      • eng_vref.txt  (English verse references — same as rel)\n#      • eng_extract.txt  (English verse text)\n# 4. After uploading, re-run the next cell (prepare_corpus)\n#\n# Quick sanity check:\n!ls -lh data/\n!wc -l data/*.txt 2>/dev/null || echo 'Files not found — upload them!'\n

In [ ]:
# Clean and split corpus into train/val/test
!python scripts/prepare_corpus.py

# Show statistics
!ls -lh data/
!wc -l data/*.txt

In [ ]:
# Train NLLB-200 fine-tuned model with LoRA
# Reduced epochs for demo — increase for production

!python scripts/train_nmt.py --batch 4 --epochs 5 --save-steps 100

# Model saved to: models/rendille-rel/

In [ ]:
# Test the trained model
!python scripts/translate.py --text "Kaayo" --model models/rendille-rel
!python scripts/translate.py --text "Yiye" --model models/rendille-rel

In [ ]:
# Launch Gradio web UI
import threading, time

def launch_demo():
    from pipeline.demo_app import main
    main()

thread = threading.Thread(target=launch_demo, daemon=True)
thread.start()
time.sleep(5)
print('='*60)
print('🎉 Gradio demo starting...')
print('   If you see a Gradio URL below, click it to open the UI')
print('='*60)

## 🔊 Speech-to-Speech (When Audio Data Arrives)

Once you obtain the Rendille Bible audio (~40 hrs) from MegaVoice,
run these additional cells:

```python
# 1. Upload audio ZIP or download from Google Drive
# Place in: asr/data/audio_raw/

# 2. Preprocess: convert MP3 → 16kHz WAV, create Whisper manifest
!python asr/preprocess_asr.py

# 3. Fine-tune Whisper ASR
!python asr/train_whisper.py --batch 4 --epochs 5

# 4. Full speech-to-speech pipeline
!python pipeline/orchestrator.py input.wav
```

**See:** `SPEECH_PIPELINE_GUIDE.md` for details